This file imports data from PG&E's GRIP, using the ArcGIS REST Endpoint, the data is saved as a geopackage
See GRIP documentation at: https://www.pge.com/assets/pge/docs/about/doing-business-with-pge/pge-grip-user-guide.pdf


In [ ]:
#Import packages
import pandas as pd
import io
import geopandas as gpd
from arcgis.gis import GIS
from arcgis.features import FeatureLayer
from arcgis.features import FeatureSet
from pathlib import Path

# Disable URL warnings
import urllib3
urllib3.disable_warnings()

#Log into AGOL anonymously
gis = GIS()

In [2]:
#set folder paths
project_fldr = Path.cwd().parent
raw_data_fldr = project_fldr / 'Data' / 'Raw'
processed_data_fldr = project_fldr / 'Data' / 'Processed'

print(project_fldr)
print(raw_data_fldr)
print(processed_data_fldr)

v:\GDA_Final_Project
v:\GDA_Final_Project\Data\Raw
v:\GDA_Final_Project\Data\Processed


In [8]:
# Fetch the feature layers 
#LGP Line layer
LGP_line_lyr = FeatureLayer("https://services2.arcgis.com/mJaJSax0KPHoCNB6/ArcGIS/rest/services/LGP_API/FeatureServer/0") 
#DRP line layer
DRP_line_lyr = FeatureLayer("https://services2.arcgis.com/mJaJSax0KPHoCNB6/ArcGIS/rest/services/DRPComplianceRelProd/FeatureServer/3")
#DRP ICA NA layer
DRP_ICA_NA_lyr = FeatureLayer("https://services2.arcgis.com/mJaJSax0KPHoCNB6/ArcGIS/rest/services/DRPComplianceRelProd/FeatureServer/4")

In [ ]:
#claude was used here to figure out error with DRP line request, there is one row with no geometry that is dropped
#be aware that any other FL read in with no geometries will auto be dropped
#create function to query feature layer & export to geopackage
def query_function(feature_layer, layer_name, division_name, out_fldr, crs):
    f_set = feature_layer.query(f"Division = '{division_name}'", out_sr=4326)    # query into a feature set object
    d = f_set.to_dict()                                             #makes the feature set a plain dict
    d['features'] = [f for f in d['features'] if f.get("geometry")] #drops rows with no geometry
    geojson_string = FeatureSet.from_dict(d).to_geojson                               #export feature set as a geojson string
    gdf = gpd.read_file(io.StringIO(geojson_string)).to_crs(crs)    #create geodf from geoJSON, convert to correct coord sys

    #save as geopackage
    gdf.to_file(out_fldr / f"{division_name.replace(' ','_')}_{layer_name}.gpkg",
                driver="GPKG",
                layer=layer_name) 

#set function inputs
out_fldr = raw_data_fldr    #output folder
division_name = 'San Jose'  #division name
crs = 26910                 #set coord system 
features_list = [LGP_line_lyr, DRP_line_lyr, DRP_ICA_NA_lyr] #list of feature layers: LGP_line_lyr, DRP_line_lyr, DRP_ICA_NA_lyr
layer_name = ['LGP_line', 'DRP_line', 'ICA_NA_line']  #layer name and used in gpkg name, match order of the feature list 'LGP_line', 'DRP_line', 'ICA_NA

#loop through query for list of features & names
for feature, name in zip(features_list, layer_name):
    query_function(feature, name, division_name,out_fldr, crs)